# W2Q2 — Time on Website Before Signup
**Question:** How much pre-signup engagement (visits, days) predicts trial conversion? Which landing pages drive the most conversions?

**Audience:** Product Team  
**Data source:** `ANALYTICS.MARTS.MART_WEB_BEHAVIOR`  
**SQL:** `sql/W2Q2_time_on_website_before_signup.sql`

## Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

from src.connection import query

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 130})
PALETTE = sns.color_palette('muted', 8)

## Load Data

In [ ]:
with open('../sql/W2Q2_time_on_website_before_signup.sql') as f:
    sql = f.read()

df = query(sql)
print(f"Rows: {len(df):,}")
df.head(3)

## Pre-Signup Visits — Converters vs Non-Converters

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics = [
    ('visits_before_signup', 'Visits Before Signup'),
    ('days_first_visit_to_signup', 'Days from First Visit to Signup'),
]

for ax, (col, label) in zip(axes, metrics):
    for val, name, color in [(True, 'Converted to Trial', PALETTE[0]), (False, 'Did Not Convert', PALETTE[1])]:
        data = df[df['converted_to_trial'] == val][col].dropna()
        data = data[data >= 0]
        ax.hist(data, bins=30, alpha=0.6, label=name, color=color, edgecolor='white')
    ax.set_title(label, fontsize=12)
    ax.set_xlabel(label)
    ax.set_ylabel('Number of Users')
    ax.legend(fontsize=9)
    sns.despine(ax=ax)

fig.suptitle('Pre-Signup Behaviour — Converters vs Non-Converters', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../output/W2Q2_pre_signup_engagement.png', bbox_inches='tight')
plt.show()

## Summary Stats — Converters vs Non-Converters

In [ ]:
metrics = ['visits_before_signup', 'days_first_visit_to_signup', 'total_visits', 'unique_pages_visited']
labels  = ['Visits Before Signup', 'Days First Visit→Signup', 'Total Visits', 'Unique Pages Visited']

rows = []
for col, label in zip(metrics, labels):
    for converted in [True, False]:
        data = df[df['converted_to_trial'] == converted][col].dropna()
        rows.append({
            'Metric': label,
            'Group': 'Converted' if converted else 'Not Converted',
            'N': len(data),
            'Median': round(data.median(), 1),
            'Mean': round(data.mean(), 1),
        })

pd.DataFrame(rows)

## Mann-Whitney U Test — Login Duration

In [ ]:
converted     = df[df['converted_to_trial'] == True]['login_duration'].dropna()
not_converted = df[df['converted_to_trial'] == False]['login_duration'].dropna()

stat, p = stats.mannwhitneyu(converted, not_converted, alternative='two-sided')
print(f"Mann-Whitney U: {stat:,.0f}  |  p-value: {p:.4f}")
print(f"Median — Converted: {converted.median():.2f}  |  Not Converted: {not_converted.median():.2f}")

## Landing Page Conversion Rates

In [ ]:
page_stats = (
    df.groupby('first_visit_page')
      .agg(
          unique_visitors=('user_id', 'count'),
          converted=('converted_to_trial', 'sum'),
      )
      .assign(conversion_rate=lambda x: (x['converted'] / x['unique_visitors'] * 100).round(1))
      .sort_values('unique_visitors', ascending=False)
      .reset_index()
)
display(page_stats)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=page_stats, x='first_visit_page', y='conversion_rate', palette='muted', ax=ax)
ax.set_title('Trial Conversion Rate by First Page Visited', fontsize=13, pad=12)
ax.set_xlabel('First Page Visited')
ax.set_ylabel('Conversion Rate (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
sns.despine()
plt.tight_layout()
plt.savefig('../output/W2Q2_landing_page_conversion.png', bbox_inches='tight')
plt.show()

## Findings

- **Visits before signup — converters vs non-converters:** ...
- **Days from first visit to signup:** ...
- **Statistical test (login duration):** ...
- **Landing page conversion rates:** ...
- **Recommendation:** ...